## Set Price regime

In [2]:
from datasets import load_dataset
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

c:\Users\lenovo\AppData\Local\Programs\Python\Python313\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


خواندن خروجی مرحله ی قبل

In [3]:
df = pd.read_feather("../Outputs/01_df.feather")

In [4]:
pd.set_option("display.max_rows", None)
pd.set_option("display.max_columns", None)
pd.set_option("display.width", None)
pd.set_option("display.max_colwidth", None)

In [5]:
df["cat2_slug"].unique()

['temporary-rent', 'residential-sell', 'residential-rent', 'commercial-rent', 'commercial-sell', 'real-estate-services']
Categories (6, object): ['commercial-rent', 'commercial-sell', 'real-estate-services', 'residential-rent', 'residential-sell', 'temporary-rent']

In [10]:
# ۱. تعریف ماسک‌های پایه
is_sell = df["cat2_slug"].isin(["commercial-sell", "residential-sell"])
is_rent = df["cat2_slug"].isin(["commercial-rent", "residential-rent"])

has_price = df["price_value"].gt(0)
has_rent = df["rent_value"].gt(0)
has_credit = df["credit_value"].gt(0)

# ۲. تعریف شرایط برای قیمت‌گذاری مستقیم
conditions = [
    # --- Invalid (داده‌های متناقض) ---
    is_sell & (df["rent_value"].gt(0) | df["credit_value"].gt(0) | df["rent_mode"].notna() | df["credit_mode"].notna()),
    is_rent & (df["price_value"].gt(0) | df["price_mode"].notna()),
    
    # --- Sale ---
    is_sell,
    
    # --- Rent & Mortgage ---
    # رهن کامل
    is_rent & (df["rent_mode"].isin(["مجانی"])) & (df["credit_mode"] == "مقطوع") & has_credit,
    # رهن و اجاره
    is_rent & (df["rent_mode"] == "مقطوع") & (df["credit_mode"] == "مقطوع") & has_rent & has_credit,
    is_rent & (df["rent_mode"] == "مقطوع") & (df["credit_mode"].isin(["توافقی"])) & has_rent,
    is_rent & (df["rent_mode"].isin(["توافقی"])) & (df["credit_mode"] == "مقطوع") & has_credit,
    is_rent & (df["rent_mode"].isin(["توافقی"])) & (df["credit_mode"] == "توافقی"),

    # فقط اجاره
    is_rent & (df["rent_mode"] == "مقطوع") & (df["credit_mode"].isin(["مجانی"])) & has_rent
]

choices = [
    "Invalid",          # Invalid
    "Invalid",          # Invalid
    "Sale",             # Sale
    "Morgage",          # Full Morgage
    "Rent and Morgage", # Rent + Morgage
    "Rent and Morgage", # Rent + Morgage
    "Rent and Morgage", # Rent + Morgage
    "Rent and Morgage", # Rent + Morgage
    "Rent"              # Rent Only
]

# اعمال مستقیم رژیم قیمتی
df["price_regime"] = np.select(conditions, choices, default="Unknown")

# ۳. تعیین نوع کاربری
df["property_type"] = np.select(
    [df["cat2_slug"].str.contains("commercial", na=False), df["cat2_slug"].str.contains("residential", na=False)],
    ["Commercial", "Residential"],
    default="Other"
)

# ۴. گزارش‌گیری سریع
print(f"تعداد کل ردیف‌ها: {len(df):,}")
print("\nتوزیع رژیم‌های قیمتی:")
print(df.groupby(["property_type", "price_regime"], dropna=False).size())


تعداد کل ردیف‌ها: 1,000,000

توزیع رژیم‌های قیمتی:
property_type  price_regime    
Commercial     Invalid                 38
               Morgage               3804
               Rent                  1935
               Rent and Morgage     70701
               Sale                 38857
               Unknown                 93
Other          Unknown              49306
Residential    Invalid                  9
               Morgage              55418
               Rent                   924
               Rent and Morgage    220178
               Sale                558705
               Unknown                 32
dtype: int64


In [11]:
invalid_sell= df[(df["price_regime"]=="Unknown") & (df["property_type"]=="Residential")]
# df[
#     (df["ad_type"] == "credit") & 
#     (df["credit_mode"] != "مقطوع")
# ]
invalid_sell[
    [
        "cat2_slug",
        "title",
        "price_mode",
        "price_value",
        "rent_mode",
        "credit_mode",
        "rent_value",
        "credit_value",
        "description"
    ]
].head(20)

,cat2_slug,title,price_mode,price_value,rent_mode,credit_mode,rent_value,credit_value,description
7527,residential-rent,مجتمع یاسمین الهیه,NaN,NaN,مجانی,توافقی,0.0,NaN,امکانات معمولی املاک سعید تخلیه 200میلیون رهن کامل
62291,residential-rent,نیازمندهمخانه. خانم,NaN,NaN,NaN,NaN,NaN,NaN,نیازمند همخانه خانم. بدون حاشیه ومرتب. لطفا اقایان تماس نگیرند. خانم های واجدشرایط فقط تماس بگیرند جهت هماهنگی. محدوده خانه. خیابان فاضل
69562,residential-rent,همخونه,NaN,NaN,NaN,NaN,NaN,NaN,همخونه میخوام
102985,residential-rent,سه خوابه /دو نبش(سازه خاص),NaN,NaN,NaN,NaN,NaN,NaN,قابل توجه کسانی که تلفیقی از نور و نقشه رو یکجا میخواهند❌ آشپزخانه جزیره✔️ آشپزخانه کثیف✔️ صفحه کورین✔️ کابینت نیو کلاسیک✔️ نورگیر سرتاسری☑️ سقف بلند☑️ دو بالکن کاربردی☑️ نقشه فوق تصور☑️ یک خواب مستر✅ بدون دیوار مشترک✅ 2 پارکینگ سندی✅ فرعی دنج❇️ دسترسی عالی❇️ اولویت با اولین تماس☎️ املاک کوروش مشاور امورملکی شما: ماهان〽️
236013,residential-rent,همخونه میخوام,NaN,NaN,NaN,NaN,NaN,NaN,همخونه خوب با اخلاق باشد
277771,residential-rent,خونه دارم همخونه میخوام,NaN,NaN,NaN,NaN,NaN,NaN,مشخصات ارسال کنین تماس میگیرم. خونه با تمامی لازم موجوده. همخونه برای مادرم
339693,residential-rent,زیر زمین 150متری اجاره,NaN,NaN,مجانی,توافقی,0.0,NaN,زیر زمین 150متری داری آب وبرق مجزا گاز مشترک ماشین رو اجاره داده می شود جهت کارگاه کابینتکاری ورنگکاری منبت کاری انبار داری مبل کاری وهرگونه شغل مناسب آبرومندانه اجاره داده می شود درحال حاضر رنگکاری میز مبل میباشد وسریع تخلیه می گردد واقع در محمدشهر کوی بهار خیابان 24متری زینبیه روبروی زینبیه 4
354152,residential-rent,بنای ماورای مسکونی،188 متری در برج مشهور الهیه,NaN,NaN,مجانی,توافقی,0.0,NaN,توجه. توجه واحد بسیار شیک تر و باشکوه تر از این تصاویر می باشد مجلل ترین رزیدنتال مسکونی در قلب منطقه یک محله ی الهیه تهران ساختاری سبز و فولدینگ با چهار جهت نمای خاص و مدرن در 17 طبقه با 114 پارکینگ دستخط آرشیتکت و مهراز خوش خط و ارزشمند ایران زمین لابی مجلل با ارتفاع کف تا سقفی 6 متری در مساحتی نزدیک به 500 متر مربع هر طبقه 4 واحد در چهار تیپ متراژ 186 متر واحدی شمال غربی با چشم اندازی بی نظیر دارای 2 پارکینگ سندی همراه با 3 خواب و هر سه مستر تراس های قابل چیدمان پنجره ها و نورگیر های تمام قد بدون OKB فول مشاعات آبی و ورزشی در مساحت های وسیع فضای این واحدی که کاملا تفکیک شده اما با پلانی باز می باشد کاملا درونگرا جهت رهن کامل با قابلیت تبدیل به استثنایی ترین قیمت ممکن
364843,residential-rent,120 متر نوساز شهید عراقی-خواجه عبدالله,NaN,NaN,مجانی,توافقی,0.0,NaN,120 متر-نوساز دو خوابه - یک خواب مستر شخصی ساز چهار طبقه تک واحدی طبقه دوم فول امکانات سالن یک تیکه و بدون پرتی TV ROOM ساختمان هوشمند نورگیر فوق العاده-بدون مشرف پنجره سقف تا کف کف سنگ اسلپ متریال برند شیرآلات توکار - وال هنگ لاندری سیستم برق هوشمند - تاچ روف گاردن پارکینک باکس
392574,residential-rent,همخانه,NaN,NaN,NaN,NaN,NaN,NaN,اقا هستم 49 سال


In [ ]:
df.to_feather("../Outputs/02_df.feather")
